In [ ]:
import numpy as np
import pandas as pd
import copy

import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

from data import fetch_dataset
from util import move_sliding_window, num_params
from model import LSTMNet, GRUNet
from algorithm import fedavg

pd.set_option('display.max_columns', None)
pd.set_option('display.max_columns', None)
np.set_printoptions(suppress=True, floatmode='fixed')

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print('running on gpu')
else:
    device = torch.device("cpu")

running on gpu
CPU times: total: 31.2 ms
Wall time: 7.98 ms


# Parameters

In [3]:
window_size = 90 # Define window_size period and split inputs/labels\
batch_size = 1024
label_col_index = 0

# seq_len = 90  # (timestamps)
hidden_dim = 256
n_layers = 2
lr = 0.001
output_dim = 1

#fed train params
num_local_epochs = 1
max_rounds = 100 #nb of total rounds for training
num_clients_per_round = 5 #number of clients to participate in training

# Data Preperation

In [4]:
dataframes = fetch_dataset()

Floor3
Floor4
Floor5
Floor6
Floor7
Floor3
Floor4
Floor5
Floor6
Floor7


In [5]:
n_clients = len(dataframes) #total number of clients to partition data into

In [6]:
for _, df in dataframes.items():
    print(df.columns)
    print(df.shape)

Index(['total_demand', 'z1_AC1(kW)', 'z1_AC2(kW)', 'z1_AC3(kW)', 'z1_AC4(kW)',
       'z1_Light(kW)', 'z1_Plug(kW)', 'z2_AC1(kW)', 'z2_Light(kW)',
       'z2_Plug(kW)', 'z3_Light(kW)', 'z3_Plug(kW)', 'z4_AC1(kW)',
       'z4_Light(kW)', 'z4_Plug(kW)', 'z5_AC1(kW)', 'z5_Light(kW)',
       'z5_Plug(kW)', 'Year', 'Month', 'Day', 'Hour', 'Minute'],
      dtype='object')
(79056, 23)
Index(['total_demand', 'z1_AC1(kW)', 'z1_AC2(kW)', 'z1_AC3(kW)', 'z1_AC4(kW)',
       'z1_Light(kW)', 'z1_Plug(kW)', 'z2_AC1(kW)', 'z2_Light(kW)',
       'z2_Plug(kW)', 'z3_Light(kW)', 'z3_Plug(kW)', 'z4_AC1(kW)',
       'z4_Light(kW)', 'z4_Plug(kW)', 'z5_AC1(kW)', 'z5_Light(kW)',
       'z5_Plug(kW)', 'Year', 'Month', 'Day', 'Hour', 'Minute'],
      dtype='object')
(79056, 23)
Index(['total_demand', 'z1_AC1(kW)', 'z1_AC2(kW)', 'z1_AC3(kW)', 'z1_AC4(kW)',
       'z1_Light(kW)', 'z1_Plug(kW)', 'z2_AC1(kW)', 'z2_Light(kW)',
       'z2_Plug(kW)', 'z3_Light(kW)', 'z3_Plug(kW)', 'z4_AC1(kW)',
       'z4_Light(kW)', '

# Build the training set

In [7]:
train_loader = []
test_loader = []
label_scalers = []

for _, df in dataframes.items():
    inputs_cols_indices = range(0, df.shape[1])  # use (total_demand,Year,Month,Day,Hour,Minute) columns as features

    #move the window
    inputs, labels = move_sliding_window(
        df.values,
        window_size,
        inputs_cols_indices=inputs_cols_indices,
        label_col_index=label_col_index
    )    
    
    # Normalize the input data columns
    sc = MinMaxScaler()
    # Obtaining the scaler for the labels(usage data) so that output can be re-scaled to actual value during evaluation
    label_sc = MinMaxScaler() 
    
    # Split data into train/test portions and combining all data into a single array
    test_portion = int(0.2 * len(inputs))
    
    train_x = sc.fit_transform(inputs[:-test_portion].reshape(-1, window_size * df.shape[1]))
    train_x = train_x.reshape(-1, window_size, df.shape[1])
    train_y = label_sc.fit_transform(labels[:-test_portion])

    test_x = sc.transform(inputs[-test_portion:].reshape(-1, window_size * df.shape[1]))
    test_x = test_x.reshape(-1, window_size, df.shape[1])
    test_y = label_sc.transform(labels[-test_portion:])
    
    # test_x.append(testx)
    # test_y.append(testy)
    label_scalers.append(label_sc)
    
    # pytorch data loaders
    train_data = TensorDataset(torch.from_numpy(train_x).to('cpu'), torch.from_numpy(train_y).to('cpu'))
    train_loader.append(DataLoader(train_data, batch_size=batch_size, drop_last=True))# Drop the last incomplete batch
    test_data = TensorDataset(torch.from_numpy(test_x).to('cpu'), torch.from_numpy(test_y).to('cpu'))
    test_loader.append(DataLoader(test_data, batch_size=batch_size))# Drop the last incomplete batch
    
    # release some memory
    del train_x, train_y

input_dim = next(iter(train_loader[0]))[0].shape[2]  # 22

(78966, 90, 23) (78966, 1)
(78966, 90, 23) (78966, 1)
(78966, 90, 23) (78966, 1)
(68823, 90, 23) (68823, 1)
(78966, 90, 23) (78966, 1)


# LSTM model

In [8]:
lstm = LSTMNet(input_dim, hidden_dim, output_dim, n_layers)
model_type = 'LSTM'
print(lstm)
print(num_params(lstm))

LSTMNet(
  (lstm): LSTM(23, 256, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (relu): ReLU()
)
814337


## FedAvg

In [9]:
%%time
lstm_K5_avg = copy.deepcopy(lstm)
outputs_avg, targets_avg, loss_avg, smape_avg, mae_avg, rmse_avg = fedavg(
    global_model = lstm_K5_avg,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    num_clients_per_round = num_clients_per_round,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
)


starting avg round 0
clients:  [2 0 1 3 4]
round 0, starting client 1/5, id: 2
Epoch [1/1], Train Loss: 0.01608482045484096
round 0, starting client 2/5, id: 0
Epoch [1/1], Train Loss: 0.016112951106643767
round 0, starting client 3/5, id: 1
Epoch [1/1], Train Loss: 0.016628157080973677
round 0, starting client 4/5, id: 3
Epoch [1/1], Train Loss: 0.015764784094227373
round 0, starting client 5/5, id: 4
Epoch [1/1], Train Loss: 0.024618522981639766

sMAPE: 112.27520462041028%
calc smape: 42.569952683514714%
MAE: 7.690946771375894
RMSE: 14.45504923036334
Average Loss:  0.009573556116051044

starting avg round 1
clients:  [2 0 1 3 4]
round 1, starting client 1/5, id: 2
Epoch [1/1], Train Loss: 0.006168027458040685
round 1, starting client 2/5, id: 0
Epoch [1/1], Train Loss: 0.004798143999994595
round 1, starting client 3/5, id: 1
Epoch [1/1], Train Loss: 0.006884347723915526
round 1, starting client 4/5, id: 3
Epoch [1/1], Train Loss: 0.0068503533894161
round 1, starting client 5/5, id: 

In [10]:
np.save('metrics/fedavg_lstm_outputs.npy', outputs_avg)
np.save('metrics/fedavg_lstm_targets.npy', targets_avg)
np.save('metrics/fedavg_lstm_loss.npy',loss_avg)
np.save('metrics/fedavg_lstm_smape.npy',smape_avg)
np.save('metrics/fedavg_lstm_mae.npy',lmae_avg)
np.save('metrics/fedavg_lstm_rmse.npy',rmse_avg)

C:\Users\User\anaconda3\envs\fed\lib\site-packages\numpy\lib\npyio.py:521: VisibleDeprecationWarning:

Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.

